# HW5: Reference Motion — From Task Space to Joint Space
**NTUT MuJoCo Course — Lecture 6**

A footstep/CoM plan does not yet tell every motor what angle to take. In this homework,
solve a small inverse-kinematics problem, inspect the given full-body Placo solver, then
build a reference-motion recording that you can export and replay.

| Part | Points | Topic |
|---|---|---|
| A | 10 | FK, Jacobian and damped inverse kinematics |
| B | 10 | Frames, velocities and motion assembly |
| C | 5 | Generation, diagnostics, JSON round trip and replay |
| D | +5 bonus | Turning and world/body velocity comparison |

### Rules
- **DO NOT USE `while` LOOPS. DO NOT ALTER TEST CELLs.**
- Implement every TODO; student functions feed the actual exported data.
- Run independently of HW4: all required robot/Placo support code is given.
- The result is a **reference motion**, not a MuJoCo torque controller or proof of balance.

In [ ]:
%matplotlib inline
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
from scipy.linalg import expm
from scipy.spatial.transform import Rotation
from IPython.display import display

for root in (Path.cwd(), *Path.cwd().parents):
    if (root / 'lecture5' / 'openduck').is_dir():
        COURSE_ROOT = root
        sys.path.insert(0, str(root / 'lecture5'))
        break
else:
    raise FileNotFoundError('Run this notebook from inside the course repository.')
from openduck import (WalkConfig, generate_reference, support_halfspaces, diagnostics,
                      print_diagnostics, validate_episode, save_episode, load_episode, frame_slices)
from openduck.visualization import plot_reference, plot_joint_motion
np.set_printoptions(precision=6, suppress=True)
print('imports ok')
from openduck.motion import FIELDS, CONVENTIONS

## Part A (10 pts): Start with a two-link leg
Use a planar two-revolute-joint chain with link lengths $l_1,l_2$, joint angles
$q=[q_1,q_2]$, and foot position
$$p(q)=\begin{bmatrix}l_1\cos q_1+l_2\cos(q_1+q_2)\\l_1\sin q_1+l_2\sin(q_1+q_2)\end{bmatrix}.$$
This diagram uses x to the right and y up; zero angles point along +x. It is a teaching
chain, not the actual Duck's joint axes or complete leg geometry.

**FK** maps joint angles to a foot position. **IK** asks which angles reach a target.
Let $e=p_{target}-p(q)$ and $J=\partial p/\partial q$. Linearize to obtain $J\Delta q\approx e$.
Near a straight leg, $J$ is nearly singular. A damped step solves
$$\min_{\Delta q}\ \tfrac12\|J\Delta q-e\|^2+\tfrac12\lambda^2\|\Delta q\|^2,$$
$$\Delta q=(J^TJ+\lambda^2I)^{-1}J^Te.$$
Use `np.linalg.solve`, not an explicit inverse. This reconnects Lecture 3's residual,
linearization and regularization to a concrete motion problem.

In [ ]:
def leg_fk(q, lengths):
    """Return a 2D foot position from two joint angles and positive link lengths."""
    # TODO: implement the formula described above.
    raise NotImplementedError("leg_fk not implemented")

In [ ]:
def leg_jacobian(q, lengths):
    """Return the analytic 2x2 Jacobian of leg_fk."""
    # TODO: implement the formula described above.
    raise NotImplementedError("leg_jacobian not implemented")

In [ ]:
def damped_ik_step(J, error, damping=1e-3):
    """Solve the damped least-squares system; damping must be positive."""
    # TODO: implement the formula described above.
    raise NotImplementedError("damped_ik_step not implemented")

### Bounded iterative solver
Return a dictionary with `q`, `history` (all joint iterates), `converged` and `residual`.
Check the residual before updating and again after the final allowed update. Limit a step's
Euclidean norm to 0.25 rad. Backtrack over at most 12 candidates $1,1/2,\ldots$ until the
position error decreases. If no step works, or the iteration budget ends, return an honest
failure report rather than claiming convergence. Do not move an unreachable target.

This small solver has no joint-limit constraints. Placo's task solver adds multiple tasks
and selected constraints using a QP; our exercise illustrates the local IK idea, not its
entire algorithm.

In [ ]:
def solve_leg_ik(target, q0, lengths=(0.11,0.11), damping=1e-3, tol=1e-8, max_iters=200):
    """Return q/history/converged/residual, including honest failure for an unreachable target."""
    # TODO: implement the formula described above.
    raise NotImplementedError("solve_leg_ik not implemented")

In [ ]:
# ------------------------------------------------------------------- TEST CELL (A)
lengths=np.array([0.11,0.11]); q=np.array([0.4,-0.7]); eps=1e-6
np.testing.assert_allclose(leg_fk([0,0],lengths),[0.22,0],atol=1e-12)
J_numeric=np.column_stack([(leg_fk(q+eps*np.eye(2)[i],lengths)-leg_fk(q-eps*np.eye(2)[i],lengths))/(2*eps) for i in range(2)])
np.testing.assert_allclose(leg_jacobian(q,lengths),J_numeric,atol=1e-9)
J=leg_jacobian([0,0],lengths); e=np.array([-0.001,0.002]); damping=1e-3
dq=damped_ik_step(J,e,damping)
# Independent augmented least-squares formulation.
oracle=np.linalg.lstsq(np.vstack([J,damping*np.eye(2)]),np.r_[e,[0,0]],rcond=None)[0]
np.testing.assert_allclose(dq,oracle,atol=1e-10)
for target,q0 in [(leg_fk([-0.3,0.7],lengths),[0.4,-0.7]),([0.219,0.005],[0.01,-0.02])]:
    result=solve_leg_ik(target,q0)
    assert result['converged'] and result['residual']<1e-8
    errors=[np.linalg.norm(target-leg_fk(qi,lengths)) for qi in result['history']]
    assert np.all(np.diff(errors)<=1e-12)
    assert np.isfinite(result['history']).all()
failed=solve_leg_ik([1,0],[0.3,-0.5],max_iters=30)
assert not failed['converged'] and failed['residual']>0.7
budget=solve_leg_ik([0.1,0.1],[0.3,-0.5],max_iters=0)
assert not budget['converged'] and len(budget['history'])==1
print('Part A tests passed.')
plt.semilogy(errors,'o-'); plt.xlabel('IK update'); plt.ylabel('Foot-position residual (m)')
plt.title('Near-singular reachable target'); plt.grid(alpha=0.3); plt.show()

### Mapping the toy IK to the actual engine (given code)
The shared `openduck/engine.py` follows this pipeline:

| Object | Role |
|---|---|
| `FootstepsPlannerRepetitive` | Future foot placements |
| `WalkPatternGenerator` | CoM, trunk and foot trajectories across supports |
| `WalkTasks` | Turn desired poses into task-space errors |
| `KinematicsSolver` | Solve joint/configuration increments and integrate them |
| posture task | Keep upper-body joints near a reference posture |

Foot orientation is constrained only on the source's selected local axes. Velocity limits
are enabled, joint-position limits remain disabled as in the reference generator, and
URDF violations are separately reported. Do not mistake a soft task weight for an absolute
constraint. All tasks and recorded states use the same timestamp in the course adapter.

Reading: [Placo: kinematics solver](https://placo.readthedocs.io/en/v0.9.10/kinematics/getting_started.html).

## Part B (10 pts): Coordinate frames and velocities
A transform $T^W_B=[R^W_B,p^W_B]$ maps body coordinates to world coordinates:
$$p^W_{foot}=p^W_B+R^W_Bp^B_{foot},\qquad
p^B_{foot}=(R^W_B)^T(p^W_{foot}-p^W_B).$$
Implement for arrays: world points `(N,K,3)` and root transforms `(N,4,4)`.
Linear and joint velocities use centered differences inside the recording and second-order
one-sided differences at both ends (`np.gradient(..., axis=0, edge_order=2)` is allowed).
Require at least three samples and positive dt.

For quaternion orientations stored as **[x,y,z,w]**, compute world angular velocity using
$$\omega^W_i\approx\frac{\operatorname{Log}(R_iR_{i-1}^T)^\vee}{\Delta t}.$$
`Rotation.as_rotvec()` already contains the rotation angle; do not multiply by the angle
again. Use the first interval also for sample zero. Normalize quaternions; the rotation
representation makes $q$ and $-q$ equivalent. Reject zero quaternions. The finite-interval
formula assumes rotation between samples is less than pi; this holds for these motions.

In [ ]:
def world_to_body(points, root_T):
    """Return (N,K,3) body-frame coordinates."""
    # TODO: implement the formula described above.
    raise NotImplementedError("world_to_body not implemented")

In [ ]:
def differentiate(values, dt):
    """Differentiate a time-major array, using second-order endpoint differences."""
    # TODO: implement the formula described above.
    raise NotImplementedError("differentiate not implemented")

In [ ]:
def world_angular_velocity(quaternions, dt):
    """Return (N,3) angular velocities from xyzw quaternions; N >= 2."""
    # TODO: implement the formula described above.
    raise NotImplementedError("world_angular_velocity not implemented")

In [ ]:
# ------------------------------------------------------------------- TEST CELL (B1)
rng=np.random.default_rng(6); N=10
T=np.repeat(np.eye(4)[None],N,axis=0)
T[:,:3,:3]=Rotation.random(N,random_state=rng).as_matrix(); T[:,:3,3]=rng.normal(size=(N,3))
body=rng.normal(size=(N,2,3)); world=T[:,None,:3,3]+np.einsum('nij,nkj->nki',T[:,:3,:3],body)
np.testing.assert_allclose(world_to_body(world,T),body,atol=1e-12)
t=np.arange(N)*0.02
np.testing.assert_allclose(differentiate(t**2,0.02),2*t,atol=1e-12)
np.testing.assert_allclose(differentiate(np.column_stack([3*t,-2*t]),0.02),np.tile([3,-2],(N,1)),atol=1e-12)
# Non-identity initial orientation catches body/world confusion; rate != 1 catches angle-squared errors.
w=np.array([0.2,-0.4,0.7]); base=Rotation.from_euler('xyz',[0.4,-0.3,0.6])
quat=(Rotation.from_rotvec(t[:,None]*w)*base).as_quat(); quat[::2]*=-1
np.testing.assert_allclose(world_angular_velocity(quat,0.02),np.tile(w,(N,1)),atol=1e-10)
np.testing.assert_allclose(world_angular_velocity(2*quat,0.02),np.tile(w,(N,1)),atol=1e-10)
for action in [lambda:differentiate([1,2],0.02),lambda:world_angular_velocity(np.zeros((2,4)),0.02)]:
    try: action()
    except ValueError: pass
    else: raise AssertionError('invalid derivative inputs accepted')
print('Part B1 tests passed.')

### Assemble a recording that another program can read
Implement `make_episode(samples)` using **your** frame/velocity functions above.

| Field | Size | Meaning |
|---|---:|---|
| root_pos / root_quat | 3 / 4 | world position / xyzw orientation |
| joints_pos | J | angles in the recorded `Joints` order |
| left_toe_pos / right_toe_pos | 3 / 3 | foot-frame origins in body coordinates |
| world_linear_vel / world_angular_vel | 3 / 3 | world root velocities |
| joints_vel | J | joint rates |
| left_toe_vel / right_toe_vel | 3 / 3 | derivatives of body-frame foot coordinates |
| foot_contacts | 2 | planned left/right labels |

Body-coordinate foot derivatives are **not** world foot velocities rotated into the body:
they also contain the effect of a moving/rotating reference frame. Preserve this source format.

Build a field dictionary in the order `FIELDS`, horizontally stack arrays, and derive offsets
by accumulating field widths; do not hardcode 16 joints or 59 columns. Return JSON-compatible
lists/numbers with the metadata below. `Times` begins at zero; the first sample is after warmup.
For this finite clip use `LoopMode='Clamp'`: the two endpoints are not guaranteed to form a
seamless loop, especially during turning.

In [ ]:
# Given metadata skeleton: inspect this before implementing make_episode.
def episode_header(samples):
    return dict(FormatVersion='openduck-course-1',Robot='open_duck_mini_v2',
                CoordinateConventions=CONVENTIONS.copy(),LoopMode='Clamp',
                FPS=samples['config']['fps'],FrameDuration=1/samples['config']['fps'],
                EnableCycleOffsetPosition=False,EnableCycleOffsetRotation=False,
                MotionWeight=1,Joints=list(samples['joint_names']),Times=samples['time'].tolist(),
                Placo=dict(samples['config'],period=samples['period']))

In [ ]:
def make_episode(samples):
    """Use your functions to build all fields, offsets, sizes, Frames and mean world velocity metadata."""
    # TODO: implement the formula described above.
    raise NotImplementedError("make_episode not implemented")

In [ ]:
# ------------------------------------------------------------------- TEST CELL (B2)
# Synthetic 3-joint recording proves that export is not hardcoded to this robot's joint count.
N=10; dt=0.02; t=np.arange(N)*dt
T=np.repeat(np.eye(4)[None],N,axis=0); T[:,0,3]=0.1*t
feet=np.repeat(T[:,None],2,axis=1); feet[:,0,1,3]=0.08; feet[:,1,1,3]=-0.08
synthetic=dict(config=dict(fps=50),period=0.4,time=t,root_T=T,
               root_quat=np.tile([0,0,0,1],(N,1)),foot_T=feet,
               joints=np.column_stack([t,2*t,-t]),joint_names=['a','b','c'],contacts=np.tile([1,1],(N,1)))
trial=make_episode(synthetic); validate_episode(trial)
a=np.asarray(trial['Frames']); sl=frame_slices(trial)
assert a.shape==(N,33)  # 27 fixed entries plus twice J
np.testing.assert_allclose(a[:,sl['world_linear_vel']],np.tile([0.1,0,0],(N,1)),atol=1e-12)
np.testing.assert_allclose(a[:,sl['joints_vel']],np.tile([1,2,-1],(N,1)),atol=1e-12)
np.testing.assert_allclose(a[:,sl['left_toe_vel']],0,atol=1e-12)
np.testing.assert_allclose(a[:,sl['world_angular_vel']],0,atol=1e-12)
print('Part B2 tests passed.')

## Part C (5 pts): Generate, inspect, export and replay
Generate the default 6-second straight reference. Verify actual vs planned FK tracking,
planned ZMP support, the original URDF joint ranges, timing and data integrity. Source settings can produce sampled ZMP
containment violations; these must be reported alongside joint-limit violations.
The source does not enforce joint-position limits: a **VIOLATED** report is a finding to
discuss, not a reason to change the model limits. Physics, contact forces and torques are
outside this generator. The collision policy excludes directly connected joint interfaces
and retains non-adjacent pairs; the head/camera reference frames are supplied. A clean
neutral-pose load does not constitute a collision check of the moving trajectory.

In [ ]:
# Given generation; your Part B functions produce the actual recording.
samples=generate_reference(WalkConfig())
report=print_diagnostics(samples)
episode=make_episode(samples)
validate_episode(episode)
output_path=save_episode(episode,COURSE_ROOT/'lecture6'/'outputs'/'open_duck_straight.json')
loaded=load_episode(output_path)
print('Saved:',output_path)
plot_reference(samples); plt.show()
sl=frame_slices(episode)
plot_joint_motion(samples,np.asarray(episode['Frames'])[:,sl['joints_vel']]); plt.show()

In [ ]:
# ------------------------------------------------------------------- TEST CELL (C)
np.testing.assert_array_equal(loaded['Frames'],episode['Frames'])
assert loaded['Joints']==samples['joint_names'] and loaded['CoordinateConventions']==CONVENTIONS
frames=np.asarray(loaded['Frames']); sl=frame_slices(loaded)
assert frames.shape==(300,59)
np.testing.assert_allclose(frames[:,sl['root_pos']],samples['root_T'][:,:3,3],atol=1e-12)
np.testing.assert_allclose(frames[:,sl['joints_pos']],samples['joints'],atol=1e-12)
np.testing.assert_array_equal(frames[:,sl['foot_contacts']],samples['contacts'])
assert report['max_com_error_m']<0.01 and report['max_foot_error_m']<0.01
from openduck import support_margin
margins=np.array([support_margin(p,poly) for p,poly in zip(samples['zmp'],samples['polygons'])])
assert int(np.sum(margins < -1e-9))==report['zmp_outside_samples']
np.testing.assert_allclose(margins.min(),report['min_zmp_margin_m'])
# Independently check the reported joint violations rather than assuming there are none.
q=samples['joints']; bounds=samples['joint_bounds']
violated={j for i,j in enumerate(samples['joint_names']) if np.any(q[:,i]<bounds[i,0]-1e-8) or np.any(q[:,i]>bounds[i,1]+1e-8)}
assert set(report['joint_violations_rad'])==violated
# Invalid recordings must fail before save/replay.
import copy
bad=copy.deepcopy(episode); bad['Frame_offset'][0]['root_quat']=0
try: validate_episode(bad)
except ValueError: pass
else: raise AssertionError('overlapping frame fields accepted')
print('Part C tests passed (recording/kinematics, not hardware feasibility).')

### See the recorded motion
The player maps joints by name and fields by metadata. Green markers indicate scheduled
support; orange markers indicate swing. Playback runs once, with controls in MeshCat.
It displays kinematic poses, not a simulation of ground reactions.

Set `ENABLE_VIEWER=False` on a headless machine. Only missing visualization imports are
handled here; data, mesh, or animation errors remain visible.

In [ ]:
ENABLE_VIEWER=os.environ.get('COURSE_HEADLESS','0')!='1'
if ENABLE_VIEWER:
    try:
        import meshcat
        from openduck.visualization import replay_episode
    except ImportError as exc:
        print('MeshCat visualization unavailable:',exc)
    else:
        viewer=replay_episode(loaded)
        display(viewer.jupyter_cell())
        print('Viewer:',viewer.url())
else:
    print('Headless mode: plots, export and all numerical tests still run.')

### Short answers (included in Part C)
1. Why does damping help near a straight leg, and what does it trade away?
2. Why must root angular velocity use a rotation increment rather than Euler-angle subtraction?
3. Are body-coordinate foot derivatives equal to world foot velocity expressed in body axes?
4. Which failed or omitted checks prevent calling this a deployable walking controller?

**PUT YOUR ANSWERS HERE**

## Part D (+5 bonus): Turn and compare frames
For a vector velocity (not a point), $v^B=(R^W_B)^Tv^W$; there is no position subtraction.
Implement the vector transform, generate a turning clip with `dtheta=0.05` rad/step, and
export it using the same Part B functions. Compare body-forward speed with world x/y speed.
The planned command is nominal; measured FK root speed need not exactly equal it.

In [ ]:
def world_vectors_to_body(vectors, root_T):
    """Transform (N,3) world vectors to body axes, without translating."""
    # TODO: implement the formula described above.
    raise NotImplementedError("world_vectors_to_body not implemented")

In [ ]:
# ------------------------------------------------------------------- TEST CELL (D, BONUS)
T=np.repeat(np.eye(4)[None],3,axis=0); T[:,:3,:3]=Rotation.from_euler('z',np.pi/2).as_matrix(); T[:,:3,3]=[100,200,300]
np.testing.assert_allclose(world_vectors_to_body(np.tile([1,0,0],(3,1)),T),np.tile([0,-1,0],(3,1)),atol=1e-12)
turning=generate_reference(WalkConfig(dtheta=0.05,duration=3))
turn_episode=make_episode(turning)
save_episode(turn_episode,COURSE_ROOT/'lecture6'/'outputs'/'open_duck_turn.json')
turn_report=print_diagnostics(turning)
sl=frame_slices(turn_episode); rows=np.asarray(turn_episode['Frames'])
v_world=rows[:,sl['world_linear_vel']]
v_body=world_vectors_to_body(v_world,turning['root_T'])
np.testing.assert_allclose(np.linalg.norm(v_body,axis=1),np.linalg.norm(v_world,axis=1),atol=1e-10)
yaw=np.unwrap(Rotation.from_quat(turning['root_quat']).as_euler('xyz')[:,2])
assert yaw[-1]-yaw[0]>0.05
plt.plot(turning['time'],v_world[:,0],label='world vx')
plt.plot(turning['time'],v_world[:,1],label='world vy')
plt.plot(turning['time'],v_body[:,0],label='body vx')
plt.xlabel('Time (s)'); plt.ylabel('Velocity (m/s)'); plt.legend(); plt.grid(alpha=0.3); plt.show()
print('Part D tests passed.')

Explain why world x velocity changes during a turn even when the body continues moving forward.
Why should this turning clip not be blindly wrapped at its last frame?

**PUT YOUR ANSWER HERE**

### Submitting
1. Fill every TODO and answer the short questions. Do not change TEST CELLs.
2. **Kernel → Restart Kernel and Run All Cells**. All required tests must pass; bonus is optional.
3. Export to HTML and print to PDF. Submit the PDF and your `.ipynb`.
4. Lecture 6 also submits the generated JSON. No GUI is needed for grading.

The model is a kinematic reference generator. Passing these tests is not a certificate of
physical balance, friction feasibility, collision freedom, or torque feasibility.